## **Setup & Load Data**

In [ ]:
# install.packages("FNN")
# install.packages("ggrepel")
# install.packages("ggallin")
library(FNN)
library(lubridate)
library(tidyverse)
library(readr)
library(stringr)
library(bigrquery)
library(parallel)
library(ggrepel)
library(ggallin)

In [ ]:
EXPORT_BUCKET = "gs://plm-pers-hba1c-thresh-wb-silky-pepper-6055"
BILLING = "wb-silky-pepper-6055"
DATA_MOUNT = "/home/jupyter/workspace/raw/vwb-aou-datasets-controlled/v8"
# CDR_STORAGE_PATH = "gs://fc-aou-datasets-controlled/v8"
# WORKSPACE_CDR = "wb-silky-artichoke-2408.C2024Q3R8"
# previous proj = "terra-vpc-sc-d3cc1fbe"

In [ ]:
# Get analysis data
system(paste0("gsutil cp ", EXPORT_BUCKET, "/data_analysis_tsh.txt ./"))
data_anal <- read_tsv("data_analysis_tsh.txt")

# Get matched cohorts
system(paste0("gsutil cp ", EXPORT_BUCKET, "/matched_cohorts_tsh.txt ./"))
knn_cohort_t <- read_tsv("matched_cohorts_tsh.txt")

# Get personalized shifts
system(paste0("gsutil cp ", EXPORT_BUCKET, "/tsh_anal_shift.txt ./"))
df <- read.table("tsh_anal_shift.txt", header = TRUE)

## **Analysis**

In [ ]:
# Definitions re-used throughout
tsh_low <- 0.4
tsh_high <- 4.0

In [ ]:
# TODO: temp filter all values to 0-5
# head(data_anal)
# head(df)
data_anal_filt <- data_anal %>%
  dplyr::group_by(person_id) %>%
  dplyr::filter(!any(tsh < 0 | tsh > 5)) %>%
  dplyr::ungroup()
dim(data_anal)
dim(data_anal_filt)
n_distinct(data_anal_filt$person_id)

#### **Measurement-Level Confusion Matrix**

TO NOTE: this preliminary version is based only on measurement thresholds, does not include ICD codes.

In [ ]:
# Think will first do this for all measures for every individual
# 1. classify each measure based on standard thresholds
# 2. classify each measure based on personalized thresholds
df_mod <- df %>%
    dplyr::mutate(
        id = as.character(format(id, scientific = FALSE, trim = TRUE))) %>%
    dplyr::mutate(
        pers_tsh_low = tsh_low - shift,
        pers_tsh_high = tsh_high - shift)

data_anal_thresh <- inner_join(
    data_anal %>% dplyr::mutate(person_id = as.character(person_id)),
    df_mod,
    by = c("person_id" = "id"))

In [ ]:
dim(data_anal_thresh)
head(data_anal_thresh)

In [ ]:
# Make confusion matrix
data_anal_thresh <- data_anal_thresh %>%
    dplyr::mutate(conf_low = case_when(
        tsh > tsh_low & tsh > pers_tsh_low ~ "tn",
        tsh > tsh_low & tsh <= pers_tsh_low ~ "fn",
        tsh <= tsh_low & tsh > pers_tsh_low ~ "fp",
        tsh <= tsh_low & tsh <= pers_tsh_low ~ "tp"
    )) %>%
    dplyr::mutate(conf_high = case_when(
        tsh < tsh_high & tsh < pers_tsh_high ~ "tn",
        tsh < tsh_high & tsh >= pers_tsh_high ~ "fn",
        tsh >= tsh_high & tsh < pers_tsh_high ~ "fp",
        tsh >= tsh_high & tsh >= pers_tsh_high ~ "tp"
    ))

In [ ]:
table(data_anal_thresh$conf_low)
table(data_anal_thresh$conf_high)

#### **Updated Individual-Level Summary**

Define true early warning: low/high personalized before standard OR never standard

Need to get each person's earliest date of:

+ low TSH
+ high TSH
+ pers low TSH
+ pers high TSH
+ normal?

In [ ]:
# Modifying code from Hayley
# Note that this removes individuals with only "normal" measurements
date_summ <- data_anal_thresh %>%
  dplyr::mutate(
    pers_low = tsh <= pers_tsh_low,
    classic_low = tsh <= tsh_low,
    pers_high = tsh >= pers_tsh_high,
    classic_high = tsh >= tsh_high
  ) %>%
  tidyr::pivot_longer(
    cols = c(pers_low, classic_low, pers_high, classic_high),
    names_to = c("type", "direction"),
    names_sep = "_",
    values_to = "flag"
  ) %>%
  dplyr::filter(flag) %>%
  dplyr::group_by(person_id, type, direction) %>%
  dplyr::summarise(first_date = min(datetime), .groups = "drop") %>%
  dplyr::mutate(date_col = paste0("tsh_", direction, "_date_", type)) %>%
  dplyr::select(person_id, date_col, first_date) %>%
  tidyr::pivot_wider(
    names_from = date_col,
    values_from = first_date)

In [ ]:
head(date_summ)

In [ ]:
# Summary
cat("Distinct individuals:", n_distinct(data_anal_thresh$person_id), "\n")
cat("Distinct individuals with only normal classic/personalized:", length(setdiff(unique(data_anal_thresh$person_id), unique(date_summ$person_id))), "\n")
cat("Distinct individuals with classic/personalized high/low TSH:", n_distinct(date_summ$person_id), "\n")

In [ ]:
indv_low <- date_summ %>%
    dplyr::filter(!(is.na(tsh_low_date_pers) & is.na(tsh_low_date_classic))) %>%
    dplyr::mutate(
        low_status = dplyr::case_when(
            !is.na(tsh_low_date_pers) & is.na(tsh_low_date_classic) ~ "pers_only",
            
            !is.na(tsh_low_date_pers) & !is.na(tsh_low_date_classic) &
                tsh_low_date_pers < tsh_low_date_classic ~ "pers_early",
            TRUE ~ "no_diff"))
dim(indv_low)

In [ ]:
indv_high <- date_summ %>%
    dplyr::filter(!(is.na(tsh_high_date_pers) & is.na(tsh_high_date_classic))) %>%
    dplyr::mutate(
        high_status = dplyr::case_when(
            !is.na(tsh_high_date_pers) & is.na(tsh_high_date_classic) ~ "pers_only",
            
            !is.na(tsh_high_date_pers) & !is.na(tsh_high_date_classic) &
                tsh_high_date_pers < tsh_high_date_classic ~ "pers_early",
            TRUE ~ "no_diff"))
dim(indv_high)

In [ ]:
# For now, not worrying about classifying the no difference further
indv_low %>%
  count(low_status) %>%
  mutate(percent = (n / sum(n))*100)

indv_high %>%
  count(high_status) %>%
  mutate(percent = (n / sum(n))*100)

#### **Example Individual Plot like CCPM**

TO NOTE: current code is set up to exactly match previous analysis by Matthew Joel
in CCPM, but should be revisited due to geom_density behavior with scale limits.

In [ ]:
head(data_anal_thresh)
head(indv_high)

In [ ]:
# Find good example
    # less than 15 measurements
    # personalized early
    # shift > 0.2?

high_plot_list <- indv_high %>%
    dplyr::filter(high_status == "pers_early") %>%
    dplyr::pull(person_id)

high_plot_df <- data_anal_thresh %>%
    dplyr::filter(person_id %in% high_plot_list) %>%
    dplyr::filter(shift > 0.75) %>%
    dplyr::group_by(person_id) %>%
    dplyr::filter(n() > 6 & n() < 15) %>%
    filter(sum(conf_high == "fn", na.rm = TRUE) >= 2) %>%
    dplyr::arrange(datetime, .by_group = TRUE) %>%
    dplyr::mutate(measure_order = paste0("v", row_number())) %>%
    dplyr::ungroup()

In [ ]:
dim(high_plot_df)
high_plot_df_disp <- high_plot_df %>% dplyr::select(-conf_low, -shift)

# high_plot_df_disp[1:60,]

In [ ]:
plot_tsh_density <- function(
    id,
    data_anal_thresh,
    data_anal,
    knn_cohort_t,
    tsh_high,
    high_plot_df,
    cohort_fill = "steelblue",
    cohort_outline = "steelblue",
    threshold_color = "#556B2F"
) {

    # Cohort IDs
    cohort_ids <- unlist(knn_cohort_t[, id], use.names = FALSE)

    # Cohort data
    group_data <- data_anal_thresh %>%
        dplyr::filter(person_id %in% cohort_ids)

    # Person-specific values
    pers_tsh_high <- unique(
        data_anal_thresh$pers_tsh_high[
            data_anal_thresh$person_id == id
        ]
    )

    pers_tsh_low <- unique(
        data_anal_thresh$pers_tsh_low[
            data_anal_thresh$person_id == id
        ]
    )

    shift_val <- unique(
        data_anal_thresh$shift[
            data_anal_thresh$person_id == id
        ]
    )

    # Density peaks
    full_d <- density(data_anal$tsh, na.rm = TRUE)
    full_d_peak <- full_d$x[which.max(full_d$y)]

    cohort_d <- density(group_data$tsh, na.rm = TRUE)
    cohort_d_peak <- cohort_d$x[which.max(cohort_d$y)]

    # Threshold lines
    vlines <- data.frame(
        x = c(tsh_high, pers_tsh_high),
        threshold = c("Classic High", "Cohort High")
    )

    options(repr.plot.width = 12, repr.plot.height = 6)

    p <- ggplot() +

        # Filled densities
        geom_density(
            data = data_anal_thresh,
            aes(x = tsh, fill = "All"),
            adjust = 2,
            alpha = 0.5,
            color = NA
        ) +

        geom_density(
            data = group_data,
            aes(x = tsh, fill = "Cohort"),
            alpha = 0.5,
            color = NA
        ) +

        # Density outlines
        geom_density(
            data = data_anal_thresh,
            aes(x = tsh),
            adjust = 2,
            color = "grey50",
            linewidth = 0.7,
            fill = NA,
            show.legend = FALSE
        ) +

        geom_density(
            data = group_data,
            aes(x = tsh),
            color = cohort_outline,
            linewidth = 0.7,
            fill = NA,
            show.legend = FALSE
        ) +

        # Threshold lines
        geom_vline(
            data = vlines,
            aes(
                xintercept = x,
                color = threshold,
                linetype = threshold
            ),
            linewidth = 0.7,
            key_glyph = draw_key_path
        ) +

        # Individual measurements
        geom_point(
            data = data_anal_thresh %>%
                dplyr::filter(person_id == id),
            aes(x = tsh, y = 0),
            color = "black",
            fill = "black",
            size = 3,
            stroke = 0.3
        ) +

        # Measurement labels
        ggrepel::geom_text_repel(
            data = high_plot_df %>%
                dplyr::filter(person_id == id),
            aes(
                x = tsh,
                y = 0,
                label = measure_order
            ),
            nudge_y = 0.03,
            size = 3,
            segment.colour = "grey60",
            max.overlaps = Inf,
            show.legend = FALSE
        ) +

        # Fill colors
        scale_fill_manual(
            name = "Distribution",
            values = c(
                "All" = "grey70",
                "Cohort" = cohort_fill
            )
        ) +

        # Threshold colors
        scale_color_manual(
            name = "Thresholds",
            breaks = c("Classic High", "Cohort High"),
            values = c(
                "Classic High" = threshold_color,
                "Cohort High" = threshold_color
            ),
            labels = c(
                paste0(
                    "Classic High ",
                    round(tsh_high, 2)
                ),
                paste0(
                    "Cohort High ",
                    round(pers_tsh_high, 2)
                )
            )
        ) +

        # Threshold line types
        scale_linetype_manual(
            name = "Thresholds",
            breaks = c("Classic High", "Cohort High"),
            values = c(
                "Classic High" = "solid",
                "Cohort High" = "dashed"
            ),
            labels = c(
                paste0(
                    "Classic High ",
                    round(tsh_high, 2)
                ),
                paste0(
                    "Cohort High ",
                    round(pers_tsh_high, 2)
                )
            )
        ) +

        scale_x_continuous(
            limits = c(0, 8),
            breaks = seq(0, 8, 0.5)
        ) +

        labs(
            title = "TSH Density: Global vs. Left-Shifted Cohort",
            subtitle = sprintf(
                "Shift: %.2f (Global Peak %.2f - Cohort Peak %.2f)",
                shift_val,
                full_d_peak,
                cohort_d_peak
            ),
            x = "TSH (mIU/L)",
            y = "Density"
        ) +

        theme_minimal(base_size = 13) +

        theme(
            legend.position = "right",
            legend.box = "vertical",
            legend.key.width = unit(1.5, "cm"),
            legend.key.height = unit(0.4, "cm")
        )

    return(p)
}

In [ ]:
p_anno_1 <- plot_tsh_density(
    id = "1499571",
    data_anal_thresh = data_anal_thresh,
    data_anal = data_anal,
    knn_cohort_t = knn_cohort_t,
    tsh_high = tsh_high,
    high_plot_df = high_plot_df)

In [ ]:
p_anno_2 <- plot_tsh_density(
    id = "8996926",
    data_anal_thresh = data_anal_thresh,
    data_anal = data_anal,
    knn_cohort_t = knn_cohort_t,
    tsh_high = tsh_high,
    high_plot_df = high_plot_df)

In [ ]:
# Show simplified plot for grant
p_simp_1 <- p_anno_1 +
    labs(title = NULL, subtitle = NULL) +
    theme(
        legend.position = "none")
p_simp_1

p_simp_2 <- p_anno_2 +
    labs(title = NULL, subtitle = NULL) +
    theme(
        legend.position = "none")
p_simp_2

In [ ]:
ggsave(
    "aou_example_trajectory_tsh_1_anno.png",
    plot = p_anno_1,
    width = 9, height = 4.5, units = "in",
    dpi = 600, device = ragg::agg_png, bg = "white")
ggsave(
    "aou_example_trajectory_tsh_2_anno.png",
    plot = p_anno_2,
    width = 9, height = 4.5, units = "in",
    dpi = 600, device = ragg::agg_png, bg = "white")


ggsave(
    "aou_example_trajectory_tsh_1.png",
    plot = p_simp_1, width = 7, height = 4.2, units = "in",
    dpi = 600, device = ragg::agg_png, bg = "white")
ggsave(
    "aou_example_trajectory_tsh_2.png",
    plot = p_simp_2, width = 7, height = 4.2, units = "in",
    dpi = 600, device = ragg::agg_png, bg = "white")

#### **Percentage Left- & Right-Shifted**

In [ ]:
shift_summ <- df_mod %>%
    dplyr::select(id, shift) %>%
    dplyr::filter(!duplicated(.)) %>%
    dplyr::mutate(shift_dir = case_when(
        shift < 0 ~ "neg",
        shift > 0 ~ "pos", 
        shift == 0 ~ "none"))
dim(shift_summ)

In [ ]:
table(shift_summ$shift_dir)

In [ ]:
# positive shift means matched cohort density shifted to lower TSH
shift_summ_pos <- shift_summ %>%
    dplyr::filter(shift_dir == "pos")
summary(shift_summ_pos$shift)

In [ ]:
# positive shift means matched cohort density shifted to lower TSH
shift_summ_neg <- shift_summ %>%
    dplyr::filter(shift_dir == "neg")
summary(shift_summ_neg$shift)

In [ ]:
# Sanity check example ID from above in this
"2072208" %in% shift_summ_pos$id
data_anal_thresh %>% dplyr::filter(person_id == "2072208")

#### **TSH Set Points by Ancestry**

In [ ]:
# TODO: temp control of which input dataset - original or filtered
data_anal <- data_anal_filt

In [ ]:
# Boxplot of all values by ancestry
unique(data_anal$ancestry_pred_other)
colnames(data_anal)

In [ ]:
tsh_all <- ggplot(data_anal, aes(x = ancestry_pred_other, y = tsh)) +
    geom_boxplot() +
    # scale_y_continuous(trans = ggallin::pseudolog10_trans) +
    theme_minimal() +
    labs(
        y = "TSH (mIU/L)",
        x = "Ancestral Population") +
    scale_x_discrete(labels = c(
        "afr" = "AFR",
        "amr" = "AMR",
        "eas" = "EAS",
        "eur" = "EUR",
        "mid" = "MID",
        "oth" = "OTHER",
        "sas" = "SAS"))
ggsave(
    "aou_tsh_set_point_all.png",
    plot = tsh_all, width = 7, height = 4.2, units = "in",
    dpi = 600, device = ragg::agg_png, bg = "white")

In [ ]:
# 1. Choose random 20 of each ancestry and plot?

In [ ]:
data_anal <- data_anal %>%
  group_by(person_id) %>%
  mutate(mean_tsh = mean(tsh)) %>%
  mutate(median_tsh = median(tsh)) %>%
  ungroup()

samp_ids <- data_anal %>%
  distinct(person_id, ancestry_pred_other) %>%
  group_by(ancestry_pred_other) %>%
  slice_sample(n = 20) %>%
  ungroup()
data_anal_samp <- data_anal %>%
  dplyr::filter(person_id %in% samp_ids$person_id)

In [ ]:
data_anal_samp$person_id <- reorder(data_anal_samp$person_id,
                             data_anal_samp$mean_tsh)

tsh_samp_20 <- ggplot(data_anal_samp, aes(person_id, tsh)) +
  geom_jitter(
    width = 0.12,
    size = 1.5) +
  stat_summary(
    fun = mean,
    geom = "point",
    shape = 15,
    size = 3,
    color = "red") +
  facet_grid(
      ~ ancestry_pred_other,
      scales = "free_x",
      space = "free_x",
      switch = "x",
      labeller = labeller(
        ancestry_pred_other = c(
          afr = "AFR",
          amr = "AMR",
          eas = "EAS",
          eur = "EUR",
          mid = "MID",
          oth = "OTHER",
          sas = "SAS"))) +
  # scale_y_continuous(
    # trans = ggallin::pseudolog10_trans) +
  labs(
    x = "Individual",
    y = "TSH (mIU/L)") +
  theme_minimal() +
  theme(
    axis.text.x = element_blank(),
    axis.ticks.x = element_blank())

ggsave(
    "aou_tsh_set_point_rand_20.png",
    plot = tsh_samp_20, width = 7, height = 4.2, units = "in",
    dpi = 600, device = ragg::agg_png, bg = "white")

In [ ]:
# 2. Force random 20 to have mean measurement between 0.5-4.0 as indirect
# check measurements are likely on correct scale?

In [ ]:
samp_ids <- data_anal %>%
  dplyr::filter(mean_tsh > 0.4 & mean_tsh < 4.0) %>% 
  distinct(person_id, ancestry_pred_other) %>%
  group_by(ancestry_pred_other) %>%
  slice_sample(n = 10) %>%
  ungroup()
data_anal_samp <- data_anal %>%
  dplyr::filter(person_id %in% samp_ids$person_id)

In [ ]:
data_anal_samp$person_id <- reorder(
    data_anal_samp$person_id,
    data_anal_samp$median_tsh)

medians <- data_anal_samp %>%
  dplyr::distinct(person_id, ancestry_pred_other, median_tsh) %>%
  dplyr::group_by(ancestry_pred_other) %>%
  dplyr::summarise(
    median = median(median_tsh, na.rm = TRUE),
    .groups = "drop")

pop_order <- medians %>%
  dplyr::arrange(median)

data_anal_samp$ancestry_pred_other <- factor(
  data_anal_samp$ancestry_pred_other,
  levels = pop_order$ancestry_pred_other)

medians <- medians %>%
  dplyr::mutate(
    ancestry_pred_other = factor(
      ancestry_pred_other,
      levels = pop_order$ancestry_pred_other))

tsh_samp_20_filt <- ggplot(data_anal_samp, aes(person_id, tsh)) +
  geom_point(
    size = 1.5,
    alpha = 0.5) +
  stat_summary(
    fun = median,
    geom = "point",
    shape = 15,
    size = 2,
    color = "red",) +
  geom_hline(
    data = medians,
    aes(yintercept = median),
    color = "red",
    linewidth = 0.7,
    inherit.aes = FALSE) +
  facet_grid(
      ~ ancestry_pred_other,
      scales = "free_x",
      space = "free_x",
      switch = "x",
      labeller = labeller(
        ancestry_pred_other = c(
          afr = "AFR",
          amr = "AMR",
          eas = "EAS",
          eur = "EUR",
          mid = "MID",
          oth = "OTHER",
          sas = "SAS"))) +
  # scale_y_continuous(
  #   trans = ggallin::pseudolog10_trans) +
  labs(
    x = "Individual",
    y = "TSH (mIU/L)") +
  theme_minimal() +
  theme(
    axis.text.x = element_blank(),
    axis.ticks.x = element_blank())

tsh_samp_20_filt

ggsave(
    "aou_tsh_set_point_rand_3.png",
    plot = tsh_samp_20_filt, width = 7, height = 4.2, units = "in",
    dpi = 600, device = ragg::agg_png, bg = "white")

In [ ]:
# TODO next
    # consider filtering to specific # measurements?
    # consider subsetting to those with biggest shifts?